In [1]:
from unittest import case

import requests

print('Hello')

Hello


# Yahoo Plots

```This is code```

Here 

In [2]:
from bs4 import BeautifulSoup
import requests

import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"

HEADERS = {'User-Agent': 'Mozilla/5.0'}
# HEADERS = {
#     "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
# }

def parse_human_readable(value):
    if value is None:
        return None
    value = value.replace(',','')
    last_char = value[-1]
    val_no_last_char = value[:-1]
    match(last_char):
        case 'K':
            return float(val_no_last_char) * 1_000
        case 'M':
            return float(val_no_last_char) * 1_000_000
        case 'B':
            return float(val_no_last_char) * 1_000_000_000
        case 'T':
            return float(val_no_last_char) * 1_000_000_000_000
        case _:
            return float(val_no_last_char)



def crypto_volumes():
    response = requests.get(URL, headers=HEADERS)
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', attrs={'class': 'yf-19xszjc'})
    rows = table.find('tbody').find_all('tr')
    #print(table)
    data = []
    for row in rows:
        #print(row)
        #print('-----------------------------')
        cols = row.find_all('td')
        symbol = cols[0].text.strip()
        #print(symbol)
        name = cols[1].text.strip()
        price = parse_human_readable(cols[4].text.strip())
        volume = parse_human_readable(cols[8].text.strip())
        market_cap = parse_human_readable(cols[6].text.strip())
        record = {
            'symbol': symbol,
            'name': name,
            'price': price,
            'volume': volume,
            'market_cap': market_cap,
        }
        data.append(record)
    #print(data)
    return pd.DataFrame(data)

def save_crypto_volume(df):
    df.to_csv('data/crypto_volumes.csv', index=False)
    df.to_excel('data/crypto_volumes.xlsx', index=False)
    df.to_json('data/crypto_volumes.json', orient='records', indent=4)


In [11]:
import plotly.express as px

def plot_volume(df):
    df_sorted = df.sort_values(by=['volume'], ascending=False)
    #df_sorted_big = df_sorted[df_sorted['volume'] > 1_000_000_000]
    df_sorted_med = df_sorted[(df_sorted['volume'] > 1_000_000_000) & (df_sorted['volume'] < 10_000_000_000)]
    fig = px.bar(df_sorted_med, x='symbol', y='volume', color='market_cap', title='Crypto Volumes - Yahoo finance', hover_data=['name', 'price', 'market_cap'])
    fig.show()


In [12]:
df = crypto_volumes()

save_crypto_volume(df)
plot_volume(df)